# Island-wide distribution network concentration, Mauritius

Ranks Village Council Areas (VCAs) and municipal wards by simulated distribution-line density (km of line per km² of land) and identifies the most concentrated units. Optionally, builds a regular grid over the island to show hotspots independent of administrative boundaries.

The distribution network can come from any line layer (by default the GridFinder output, `MUguess.gpkg`) or from a built mu-star product such as `inferred-provided-mauritius-rodrigues`. See the configuration cell.

Source layers and exported files are in EPSG:4326. Lengths and areas are computed in EPSG:32740 (UTM zone 40S), in memory only.

**Outputs** (under `data/out/energy/island_concentration/`)

- `admin_density_ranked.csv`: line length, land area and density for every unit, ranked
- `admin_density.gpkg`: the same, as polygons for QGIS
- `top5_density.png`, `density_distribution.png`, `top5_total_km.png`: figures
- `island_grid.gpkg`: hotspot grid (Part 3, optional)

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import box

pd.set_option("display.width", 140)

In [ ]:
for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers is the dev-only module shared by the energy notebooks: it knows where the pipeline keeps its data.
import _helpers as h  # noqa: E402

## Configuration

Paths default to the mu-star data folder (`data_root` in `config/config.yaml`). Place the input files there, or point the paths anywhere else on your machine.

- `DIST_SOURCE`: either a line file (GeoPackage, GeoJSON, shapefile or GeoParquet) or the name of a built mu-star product. For a product, the CEB transmission backbone is dropped so that only distribution edges are counted.
- `ADMIN_LAYER`: VCA and municipal ward polygons. Non-polygon features (for example label points) are ignored.
- `LAND_BOUNDARY`: island outline used to clip coastal units, so they are not credited with offshore area. Set to `None` to skip clipping.

In [ ]:
DATA_ROOT = h.DATA_ROOT

DIST_SOURCE = DATA_ROOT / "incoming" / "energy" / "gridfinder" / "MUguess.gpkg"
# DIST_SOURCE = "inferred-provided-mauritius-rodrigues"   # or a built mu-star product
DIST_LAYER = None  # layer name inside a multi-layer file; None = first layer

ADMIN_LAYER = DATA_ROOT / "incoming" / "admin" / "mauritius_vca_wards.geojson"
NAME_FIELD = None  # column holding the unit name; None = auto-detect
LAND_BOUNDARY = DATA_ROOT / "incoming" / "admin" / "mauritius_land.geojson"  # None = no clipping

MIN_AREA_KM2 = 2.0  # units smaller than this are excluded from the top five; 0 disables
RUN_GRID = True  # Part 3: island-wide hotspot grid
CELL_SIZE_M = 1000  # grid cell size (m)

METRIC_CRS = "EPSG:32740"
OUT_DIR = DATA_ROOT / "out" / "energy" / "island_concentration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Helper functions

In [ ]:
def require(path, what):
    """Stop with a clear message if an input file is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{what} not found: {path}\nPlace the file there or edit the path above.")
    return path


def read_vector(path, layer=None):
    """Read any vector file into a GeoDataFrame, assuming EPSG:4326 if no CRS is set."""
    path = require(path, "input layer")
    if path.suffix in (".parquet", ".geoparquet"):
        gdf = gpd.read_parquet(path)
    else:
        gdf = gpd.read_file(path, layer=layer)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    return gdf


def load_lines(source, layer=None):
    """Distribution lines from a file or from a built mu-star product, in the metric CRS."""
    if isinstance(source, str) and source in h.available_products():
        _, edges = h.load_layers(source)
        if "source" in edges.columns:
            print("edge sources:", edges["source"].value_counts(dropna=False).to_dict())
            backbone = edges["source"].astype(str).str.contains("transmission", case=False)
            print(f"dropping {int(backbone.sum())} transmission backbone edges")
            edges = edges[~backbone]
        gdf = edges
    else:
        gdf = read_vector(source, layer)
    gdf = gdf[gdf.geom_type.isin(["LineString", "MultiLineString"])]
    return gdf.to_crs(METRIC_CRS)[["geometry"]].reset_index(drop=True)


def dissolve_all(gdf):
    """Single geometry for the whole layer (union_all on GeoPandas >= 1.0)."""
    try:
        return gdf.union_all()
    except AttributeError:
        return gdf.unary_union


def pick_name_field(gdf):
    """Guess which column holds the unit name."""
    if NAME_FIELD:
        return NAME_FIELD
    preferred = [
        "NAME",
        "Name",
        "name",
        "VCA",
        "VCA_NAME",
        "ADM3_EN",
        "ADM2_EN",
        "shapeName",
        "LOCALITY",
        "Locality",
        "WARD",
        "Ward",
    ]
    for c in preferred:
        if c in gdf.columns:
            return c
    for c in gdf.columns:  # fall back to the first text column with mostly unique values
        if c != "geometry" and gdf[c].dtype == object and gdf[c].nunique() > 0.8 * len(gdf):
            return c
    raise RuntimeError(f"Could not identify a name column. Columns: {list(gdf.columns)}")


def line_km_per_poly(polys, lines, id_col, chunk=2000):
    """Length of `lines` (km) inside each polygon, keyed by `id_col`. Chunked and spatially indexed."""
    sidx = lines.sindex
    parts = []
    for i in range(0, len(polys), chunk):
        g = polys.iloc[i : i + chunk]
        try:
            hit = sidx.query(g.geometry, predicate="intersects")
        except TypeError:  # older GeoPandas
            hit = sidx.query_bulk(g.geometry, predicate="intersects")
        idx = np.unique(hit[1])
        if idx.size == 0:
            continue
        sub = lines.iloc[idx][["geometry"]]
        inter = gpd.overlay(sub, g[[id_col, "geometry"]], how="intersection", keep_geom_type=True)
        if len(inter):
            inter["km"] = inter.length / 1000
            parts.append(inter.groupby(id_col)["km"].sum())
    if not parts:
        return pd.Series(dtype="float64", name="km")
    return pd.concat(parts).groupby(level=0).sum()


def fishnet(poly, size):
    """Regular grid of `size` metre cells covering `poly`, clipped to it."""
    minx, miny, maxx, maxy = poly.bounds
    xs = np.arange(minx, maxx, size)
    ys = np.arange(miny, maxy, size)
    cells = [box(x, y, x + size, y + size) for x in xs for y in ys]
    grid = gpd.GeoDataFrame(geometry=cells, crs=METRIC_CRS)
    grid = grid[grid.intersects(poly)]  # drop cells wholly offshore
    grid = gpd.clip(grid, poly)  # trim coastal cells
    grid = grid[~grid.geometry.is_empty].reset_index(drop=True)
    grid["cell_km2"] = grid.area / 1e6
    return grid

## Load layers

In [ ]:
dist = load_lines(DIST_SOURCE, DIST_LAYER)

admin = read_vector(ADMIN_LAYER).to_crs(METRIC_CRS)
admin = admin[admin.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

if LAND_BOUNDARY is not None:
    island = dissolve_all(read_vector(LAND_BOUNDARY).to_crs(METRIC_CRS))
else:
    island = dissolve_all(admin)

print(f"{len(dist):,} distribution line features, {dist.length.sum() / 1000:,.0f} km in total")
print(f"{len(admin):,} administrative polygons")

## Prepare the administrative units

Units with the same name are merged, then clipped to the island outline.

In [ ]:
name_col = pick_name_field(admin)
print("using name column:", name_col)

admin = admin[[name_col, "geometry"]].rename(columns={name_col: "unit"})
admin["unit"] = admin["unit"].astype(str).str.strip()
admin = admin.dissolve(by="unit", as_index=False)

admin["geometry"] = admin.geometry.intersection(island)
admin = admin[~admin.geometry.is_empty].reset_index(drop=True)
admin["unit_km2"] = admin.area / 1e6

print(f"{len(admin)} units after cleaning, {admin.unit_km2.sum():,.0f} km2 in total")
admin[["unit", "unit_km2"]].sort_values("unit_km2", ascending=False).head()

# Part 1: Density by administrative unit

In [ ]:
km = line_km_per_poly(admin, dist, "unit")

res = admin[["unit", "unit_km2"]].merge(km.rename("km"), left_on="unit", right_index=True, how="left")
res["km"] = res["km"].fillna(0.0)
res["km_per_km2"] = res.km / res.unit_km2
res = res.sort_values("km_per_km2", ascending=False).reset_index(drop=True)
res["rank"] = res.index + 1

res.to_csv(OUT_DIR / "admin_density_ranked.csv", index=False)
print(f"distribution line inside the units: {res.km.sum():,.0f} km")
res[["rank", "unit", "km", "unit_km2", "km_per_km2"]].head(15).round(2)

## The five most concentrated units

Small units can top a density ranking on a few kilometres of line. The cell prints the top five with and without the `MIN_AREA_KM2` guard; if the two lists differ, report the guarded list and state the threshold in the methods.

In [ ]:
top5_raw = res.head(5)
top5 = res[res.unit_km2 >= MIN_AREA_KM2].head(5).reset_index(drop=True)

cols = ["rank", "unit", "km", "unit_km2", "km_per_km2"]
print("No area guard:")
print(top5_raw[cols].round(2).to_string(index=False))
print(f"\nGuard: unit_km2 >= {MIN_AREA_KM2}")
print(top5[cols].round(2).to_string(index=False))

## Bar chart

The total line length is printed beside each bar, so a reader can tell whether a high density comes from a lot of line or from a small unit.

In [ ]:
d = top5.iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(d.unit, d.km_per_km2, color="#b2182b")
for y, (v, tot) in enumerate(zip(d.km_per_km2, d.km)):
    ax.text(v, y, f"  {v:.1f} km/km2  ({tot:,.0f} km)", va="center", fontsize=9)
ax.set_xlabel("Distribution line density (km/km2)")
ax.set_title("Five most concentrated administrative units, Mauritius")
ax.set_xlim(0, d.km_per_km2.max() * 1.35)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "top5_density.png", dpi=300)
plt.show()

## Export for QGIS

Style as a graduated choropleth on `km_per_km2`; `is_top5` marks the five most concentrated units for labelling.

In [ ]:
out = admin.merge(res[["unit", "km", "km_per_km2", "rank"]], on="unit")
out["is_top5"] = out.unit.isin(top5.unit).astype(int)
out.to_crs("EPSG:4326").to_file(OUT_DIR / "admin_density.gpkg", layer="admin_density", driver="GPKG")
print("written:", OUT_DIR / "admin_density.gpkg")

# Part 2: Spread of density across units

How far the top five sit above the other units.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(res.km_per_km2, bins=40, color="#999999")
for v in top5.km_per_km2:
    ax.axvline(v, color="#b2182b", lw=1)
ax.set_xlabel("Distribution line density (km/km2)")
ax.set_ylabel("Number of administrative units")
ax.set_title("Density across all units (top five marked)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "density_distribution.png", dpi=300)
plt.show()

res.km_per_km2.describe().round(2)

# Part 3: Island-wide hotspot grid (optional)

A regular grid of `CELL_SIZE_M` cells over the island, independent of administrative boundaries. At 1,000 m this is about 2,000 cells and runs in under a minute; smaller cells take longer. Set `RUN_GRID = False` to skip.

In [ ]:
if RUN_GRID:
    grid = fishnet(island, CELL_SIZE_M)
    grid["cell_id"] = grid.index
    print(f"{len(grid):,} cells")

    km_cell = line_km_per_poly(grid, dist, "cell_id")
    grid = grid.join(km_cell.rename("dist_km"), on="cell_id")
    grid["dist_km"] = grid.dist_km.fillna(0)
    grid["dist_km_per_km2"] = grid.dist_km / grid.cell_km2

    full_cell = (CELL_SIZE_M / 1000) ** 2
    grid["is_full"] = (grid.cell_km2 >= 0.5 * full_cell).astype(int)  # 0 = coastal part-cell

    grid.to_crs("EPSG:4326").to_file(OUT_DIR / "island_grid.gpkg", layer=f"cells_{CELL_SIZE_M}m", driver="GPKG")
    print("written:", OUT_DIR / "island_grid.gpkg")
    print(grid["dist_km_per_km2"].describe().round(2))

### QGIS styling for the hotspot grid

1. Symbology: Graduated on `dist_km_per_km2`, Natural Breaks (Jenks), 5 classes.
2. Symbol: Simple Fill, stroke style **No Line**; layer opacity about 70%.
3. Filter `"is_full" = 1` to exclude part-cells at the coast, then classify again.

# Part 4: Context

Where the main towns sit in the density ranking, and which units hold the most line in absolute terms.

In [ ]:
towns = ["Port Louis", "Curepipe", "Quatre Bornes", "Beau Bassin / Rose Hill", "Vacoas-Phoenix"]
res[res.unit.str.contains("|".join(towns), case=False, na=False)][
    ["rank", "unit", "km", "unit_km2", "km_per_km2"]
].round(2)

In [ ]:
top5_km = res.sort_values("km", ascending=False).head(5).reset_index(drop=True)
print(top5_km[["unit", "km", "unit_km2", "km_per_km2", "rank"]].round(2).to_string(index=False))

d = top5_km.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(d.unit, d.km, color="#2166ac")
for y, (v, dens) in enumerate(zip(d.km, d.km_per_km2)):
    ax.text(v, y, f"  {v:,.0f} km  ({dens:.1f} km/km2)", va="center", fontsize=9)
ax.set_xlabel("Total distribution line length (km)")
ax.set_title("Five administrative units with the greatest total network length")
ax.set_xlim(0, d.km.max() * 1.35)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "top5_total_km.png", dpi=300)
plt.show()